In [1]:
import platform
print(platform.processor())
print(platform.machine())

arm
arm64


In [2]:
import torch
print(f"Torch version : {torch.__version__}")
print(f"MPS available : {torch.backends.mps.is_available()}")
print(f"MPS built     : {torch.backends.mps.is_built()}")


import transformers
print(f"Transformers: {transformers.__version__}")

Torch version : 2.11.0
MPS available : True
MPS built     : True
Transformers: 5.7.0


In [3]:
import sys
sys.path.insert(0, ".")

import json
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM

In [4]:
# tokenizer = AutoTokenizer.from_pretrained(model_id, revision="2024-08-26")
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     trust_remote_code=True,
#     revision="2024-08-26",
# ).to(device)
# model.eval()

In [5]:
# Cell 3 — Load BLIP base
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)
model.eval()
print("Model loaded ✅")

Using device: mps


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Model loaded ✅


In [7]:
# Cell 4 — Test on a single frame
import cv2
import json
from PIL import Image

with open("output/manifest.jsonl") as f:
    entry = json.loads(f.readline())

test_frame_path = entry["segments"][0]["frame_paths"][0]
print(f"Testing on  : {test_frame_path}")
print(f"Ground truth: {entry['segments'][0]['sentence']}")

# Load image
img_bgr = cv2.imread(test_frame_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
image   = Image.fromarray(img_rgb)

# Run BLIP
inputs = processor(image, return_tensors="pt").to(device)
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        repetition_penalty=1.5,   # penalizes repeating tokens
        num_beams=5,              # beam search instead of greedy
        early_stopping=True,
    )
caption = processor.decode(output[0], skip_special_tokens=True)
print(f"BLIP caption: {caption}")

Testing on  : output/frames/v_QOlSCBRmfWY/seg_000/frame_000.jpg
Ground truth: A young woman is seen standing in a room and leads into her dancing.
BLIP caption: a woman in a black bathingsuit standing on a wooden floor


In [10]:
# Test if meaningful variations are acheived cross frames in a single segment
seg = entry["segments"][1]   # longer segment, more frames
print(f"Segment: [{seg['timestamp_start']:.1f}s → {seg['timestamp_end']:.1f}s]")
print(f"Ground truth: {seg['sentence']}")
print()

for frame_path, ft in zip(seg["frame_paths"], seg["frame_timestamps"]):
    img_bgr = cv2.imread(frame_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    image   = Image.fromarray(img_rgb)

    inputs = processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=50,
            repetition_penalty=3.0,
            num_beams=5,
            early_stopping=True,
        )
    caption = processor.decode(output[0], skip_special_tokens=True)
    print(f"  {ft:.1f}s : {caption}")

Segment: [17.4s → 60.8s]
Ground truth: The girl dances around the room while the camera captures her movements.

  17.4s : a woman doing a handstant on the floor
  23.6s : a woman is doing a handstant on the floor
  29.8s : a woman is sitting on the floor in an empty room
  36.0s : a woman is dancing in an empty room
  42.2s : a person doing a handstant in a room
  48.4s : a woman in a black leo leo leo leo leo leo leo leo leo leo leo leo leo leo leo leo
  54.6s : a woman sitting on the floor in an empty room
  60.8s : a woman is dancing in an empty room


### Test Full Pipeline